# Macro research agent with FXMacroData and Mistral function calling

Build a small research loop where a Mistral model calls the FXMacroData REST API to answer questions about economic releases.

*Author: Robert Tidball ([@roberttidball](https://github.com/roberttidball)), FXMacroData*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistralai/cookbook/blob/main/third_party/FXMacroData/fxmacrodata_macro_research_agent.ipynb)

Questions like "Is US inflation still rising?" need two things a model doesn't have: the latest official figures and the date of the next release. In this cookbook, you give the model three tools that read both from the FXMacroData API, then run a function-calling loop until the model has enough data to answer.

The three tools are:

- `get_indicator_history`: recent releases for one indicator, with the time each figure was published
- `get_release_calendar`: scheduled release dates and times
- `get_catalogue`: the indicator slugs available for a currency

USD data works without an FXMacroData API key, so you only need a Mistral API key to run the whole notebook.

## Prerequisites

To complete this cookbook, you will need:

- Python 3.9 or later
- A Mistral account and API key
- Optionally, an FXMacroData API key for currencies other than USD and for full history

## Environment setup

This section installs the packages and sets your API keys.

### Install

Install the Mistral Python SDK and `requests`:

In [ ]:
%pip install -q mistralai==2.9.4 requests==2.34.2

### Required environment variables

To complete this cookbook, you'll need a Mistral API key. In [Studio](https://console.mistral.ai), navigate to the [API keys section](https://console.mistral.ai/home?profile_dialog=api-keys) and create a new API key.

You have two options:

1. Set `MISTRAL_API_KEY` as an environment variable before you start Jupyter.
2. Leave it unset and enter the key when the next cell prompts you.

In [ ]:
import getpass
import os

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

The FXMacroData key is optional. Without one, the API serves USD data. If `FXMACRODATA_API_KEY` is set, the next cell sends it in the `X-API-Key` header, which unlocks the other currencies and full history. Keep the key out of URLs and query strings.

In [1]:
import json
import os

import requests

BASE_URL = "https://api.fxmacrodata.com/v1"

HEADERS = {"Accept": "application/json"}
if os.environ.get("FXMACRODATA_API_KEY"):
    HEADERS["X-API-Key"] = os.environ["FXMACRODATA_API_KEY"]

print("FXMacroData key set:", "X-API-Key" in HEADERS)

FXMacroData key set: False


## Understand release timestamps

Each data row carries two different dates:

- `date` is the end of the reference period. August CPI has `date` 2026-08-31.
- `announcement_datetime` is when the figure was published, in Unix seconds. `announcement_datetime_local` shows the same moment in the publisher's time zone.

When you line macro data up against prices or build a backtest, use the release timestamp, not the reference date. A figure for August wasn't public until it was released in September, so using it any earlier leaks future information into the test.

Without an FXMacroData key, three limits apply:

- USD only
- the most recent 90 days of history, so you see two or three monthly releases
- releases appear 15 minutes after publication

The `freemium_window` and `freemium_delay` fields in each history response tell you when these limits applied, and the tool below passes them on to the model.

## Define the tools

Each tool is a plain Python function that calls one endpoint and trims the response to the fields the model needs. Trimming keeps tool results short, and the field names stay the same as in the API so you can check them against the [API docs](https://api.fxmacrodata.com/docs).

Two slugs need care:

- US CPI is `inflation`, not `cpi`.
- `non_farm_payrolls` is the total employment level in persons. The monthly change is the difference between two consecutive releases.

In [2]:
def _get(path, params=None):
    response = requests.get(f"{BASE_URL}{path}", params=params, headers=HEADERS, timeout=30)
    try:
        body = response.json()
    except ValueError:
        body = {"detail": response.text[:500]}
    if response.status_code != 200:
        return {"status_code": response.status_code, **{k: body[k] for k in ("code", "detail") if k in body}}
    return body


def _pick(record, fields):
    return {k: record[k] for k in fields if k in record}


ROW_FIELDS = (
    "date",
    "val",
    "announcement_datetime",
    "announcement_datetime_local",
    "previous_value",
    "change_from_previous",
)


def get_indicator_history(currency, indicator, limit=6):
    """Most recent releases for one indicator, newest first."""
    body = _get(
        f"/announcements/{currency.lower()}/{indicator}",
        params={"limit": max(1, min(int(limit), 24))},
    )
    if "data" not in body:
        return body
    result = _pick(body, ("currency", "indicator", "name", "source", "seasonal_adjustment", "value_metadata"))
    result["data"] = [_pick(row, ROW_FIELDS) for row in body["data"]]
    if body.get("freemium_window", {}).get("applied"):
        result["freemium_window"] = _pick(body["freemium_window"], ("applied", "max_days", "cutoff_date"))
    if body.get("freemium_delay", {}).get("applied"):
        result["freemium_delay"] = _pick(body["freemium_delay"], ("applied", "delay_minutes", "withheld_count"))
    return result


def get_release_calendar(currency, indicator=None):
    """Next scheduled releases for a currency, optionally for one indicator."""
    params = {"indicator": indicator} if indicator else None
    body = _get(f"/calendar/{currency.lower()}", params=params)
    if "data" not in body:
        return body
    fields = (
        "release",
        "name",
        "reference_period",
        "announcement_datetime_utc",
        "announcement_datetime_local",
        "release_date_confirmed",
        "event_importance",
    )
    return {
        "currency": body.get("currency"),
        "timezone": body.get("timezone"),
        "data": [_pick(event, fields) for event in body["data"][:10]],
    }


def get_catalogue(currency):
    """Indicator slugs available for a currency, with name, unit, and frequency."""
    body = _get(f"/data_catalogue/{currency.lower()}")
    if "status_code" in body:
        return body
    return {
        slug: _pick(meta, ("name", "unit", "frequency", "source"))
        for slug, meta in body.items()
        if isinstance(meta, dict)
    }

### Call the tools directly

Before handing the tools to the model, call each one to see what the model will receive. These calls use the keyless USD access.

In [3]:
print(json.dumps(get_indicator_history("USD", "inflation", limit=3), indent=2))

{
  "currency": "USD",
  "indicator": "inflation",
  "name": "Inflation (CPI)",
  "source": "BLS",
  "seasonal_adjustment": "NSA",
  "value_metadata": {
    "source_unit": "%YoY",
    "normalization_applied": true
  },
  "data": [
    {
      "date": "2026-08-31",
      "val": 3.4,
      "announcement_datetime": 1789129800,
      "announcement_datetime_local": "2026-09-11T08:30:00-04:00",
      "previous_value": 3.4,
      "change_from_previous": 0.0
    },
    {
      "date": "2026-07-31",
      "val": 3.4,
      "announcement_datetime": 1786537800,
      "announcement_datetime_local": "2026-08-12T08:30:00-04:00"
    }
  ],
  "freemium_window": {
    "applied": true,
    "max_days": 90,
    "cutoff_date": "2026-07-07"
  },
  "freemium_delay": {
    "applied": true,
    "delay_minutes": 15,
    "withheld_count": 0
  }
}


The calendar returns upcoming releases in time order. Filter it with an indicator slug:

In [4]:
print(json.dumps(get_release_calendar("USD", "inflation"), indent=2))

{
  "currency": "USD",
  "timezone": "America/New_York",
  "data": [
    {
      "release": "inflation",
      "name": "Inflation (CPI)",
      "reference_period": "September 2026",
      "announcement_datetime_utc": "2026-10-14T12:30:00+00:00",
      "announcement_datetime_local": "2026-10-14T08:30:00-04:00",
      "release_date_confirmed": true,
      "event_importance": "high"
    },
    {
      "release": "inflation",
      "name": "Inflation (CPI)",
      "reference_period": "October 2026",
      "announcement_datetime_utc": "2026-11-10T13:30:00+00:00",
      "announcement_datetime_local": "2026-11-10T08:30:00-05:00",
      "release_date_confirmed": true,
      "event_importance": "high"
    },
    {
      "release": "inflation",
      "name": "Inflation (CPI)",
      "reference_period": "November 2026",
      "announcement_datetime_utc": "2026-12-10T13:30:00+00:00",
      "announcement_datetime_local": "2026-12-10T08:30:00-05:00",
      "release_date_confirmed": true,
      "even

The catalogue lists every slug the history and calendar tools accept. The model uses it when a question names an indicator it hasn't seen before:

In [5]:
catalogue = get_catalogue("USD")
print(len(catalogue), "USD indicators")
for slug in ("inflation", "core_inflation", "non_farm_payrolls", "unemployment"):
    print(slug, catalogue[slug])

72 USD indicators
inflation {'name': 'Inflation (CPI)', 'unit': '%YoY', 'frequency': 'Monthly', 'source': 'BLS'}
core_inflation {'name': 'Core Inflation', 'unit': '%YoY', 'frequency': 'Monthly', 'source': 'BLS'}
non_farm_payrolls {'name': 'Nonfarm Payrolls', 'unit': 'Persons', 'frequency': 'Monthly', 'source': 'BLS'}
unemployment {'name': 'Unemployment Rate', 'unit': '%', 'frequency': 'Monthly', 'source': 'BLS'}


An unknown slug returns the API's error message, which points back to the catalogue. Passing errors through as tool results lets the model correct itself:

In [6]:
print(get_indicator_history("USD", "cpi"))

{'status_code': 404, 'detail': 'Unsupported currency (USD) or indicator (cpi). Did you mean: ppi? GET /v1/data_catalogue/usd lists every indicator slug served for USD.'}


## Describe the tools for the model

The model sees each tool as a JSON schema with a name, a description, and typed parameters. The descriptions carry the slug hints from above, since the model reads them when it picks arguments.

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_indicator_history",
            "description": (
                "Recent official releases for one macroeconomic indicator, newest first. "
                "Each row has the reference period end (date), the value (val), and the "
                "publication time (announcement_datetime_local). US CPI uses the slug "
                "'inflation'. 'non_farm_payrolls' is the employment level in persons."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "currency": {"type": "string", "description": "Three-letter currency code, for example USD."},
                    "indicator": {"type": "string", "description": "Indicator slug from get_catalogue, for example inflation."},
                    "limit": {"type": "integer", "description": "Number of releases to return, 1 to 24."},
                },
                "required": ["currency", "indicator"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_release_calendar",
            "description": "Upcoming scheduled releases for a currency, optionally filtered to one indicator slug.",
            "parameters": {
                "type": "object",
                "properties": {
                    "currency": {"type": "string", "description": "Three-letter currency code, for example USD."},
                    "indicator": {"type": "string", "description": "Optional indicator slug, for example inflation."},
                },
                "required": ["currency"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_catalogue",
            "description": "List the indicator slugs available for a currency, with name, unit, frequency, and source.",
            "parameters": {
                "type": "object",
                "properties": {
                    "currency": {"type": "string", "description": "Three-letter currency code, for example USD."},
                },
                "required": ["currency"],
            },
        },
    },
]

names_to_functions = {
    "get_indicator_history": get_indicator_history,
    "get_release_calendar": get_release_calendar,
    "get_catalogue": get_catalogue,
}

## Build the agent loop

The loop sends the conversation and the tool schemas to the model. When the response contains tool calls, it runs each function, appends the result as a `tool` message, and calls the model again. When the response has no tool calls, that message is the answer.

The system prompt asks the model to cite release times and to say when the 90-day keyless window limits the history it can see.

In [ ]:
from mistralai.client import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
model = "mistral-medium-latest"

SYSTEM_PROMPT = (
    "You answer questions about macroeconomic data using the tools provided. "
    "If you aren't sure of an indicator slug, call get_catalogue first. "
    "Base every number on tool results, and give the release time "
    "(announcement_datetime_local) for each figure you cite. "
    "If a result has a freemium_window, say that history is limited to that window."
)


def run_agent(question, max_turns=8):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    for _ in range(max_turns):
        response = client.chat.complete(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        message = response.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = tool_call.function.arguments
            if isinstance(args, str):
                args = json.loads(args)
            print(f"tool call: {name}({args})")
            result = names_to_functions[name](**args)
            messages.append({
                "role": "tool",
                "name": name,
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            })
    return "No final answer after max_turns tool rounds."

## Ask a question

Ask about the recent trend and the next release. The model usually calls `get_indicator_history` and `get_release_calendar`, and it may call `get_catalogue` first to confirm the slug:

In [ ]:
question = "How has US inflation trended over the last few releases, and when is the next CPI release?"
print(run_agent(question))

Try a question that needs a slug lookup, such as "What did the last two payrolls releases add in jobs?" The model has to find `non_farm_payrolls` and subtract consecutive levels to get the monthly change.

## Summary

This cookbook connected a Mistral model to official macroeconomic release data through three function-calling tools and a loop that runs tool calls until the model returns an answer.

**What you built**

- Three Python tools that read indicator history, the release calendar, and the indicator catalogue
- JSON schemas that describe those tools to the model
- An agent loop that returns tool results as `tool` messages until the model gives a final answer

**Mistral features used**

- Chat Completions API
- Function calling with `tool_choice="auto"`

**Other services**

- FXMacroData REST API (keyless for USD)

Read more about function calling in the [documentation](https://docs.mistral.ai/capabilities/function_calling).